# Replicating URRSM with geomfum + benchfum

This notebook shows how to use **geomfum** and **benchfum** to exactly replicate  
[Unsupervised Learning of Robust Spectral Shape Matching (URRSM, SIGGRAPH 2023)](https://arxiv.org/abs/2304.14419).

We cover:
1. Building the `RobustFMNet` model from the URRSM JSON config
2. Loading a pretrained checkpoint (trained with the original URRSM code)
3. Running inference via `TrainedModelWrapper`
4. Test-time refinement via `TestTimeRefiner`
5. Training from scratch with the URRSM training config

In [2]:
import os

# Must be set BEFORE any geomfum / geomstats imports
os.environ["GEOMSTATS_BACKEND"] = "pytorch"

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Dataset

We use the FAUST dataset.  
For a **quick smoke test** we use `dummy_faust` (2 shapes, already available in the repo).  
Switch `USE_DUMMY` to `False` and point the paths to the full FAUST dataset for real experiments.

In [ ]:
from geomfum.dataset.torch import PairsDataset, ShapeDataset

USE_DUMMY = True  # flip to False to use the full FAUST dataset

if USE_DUMMY:
    TEST_SET_PATH = "../../../datasets/dummy_datasets/dummy_faust/test_set/"
    TRAIN_SET_PATH = "../../../datasets/dummy_datasets/dummy_faust/train_set/"
else:
    TEST_SET_PATH = "../../../datasets/faust/test_set/"
    TRAIN_SET_PATH = "../../../datasets/faust/train_set/"

# URRSM uses k=200 eigenvectors (dataset level), WKS computed from the first 128
K_EIG = 200

test_shapes = ShapeDataset(
    TEST_SET_PATH,
    spectral=True,
    distances=True,  # needed for geodesic error
    correspondences=True,  # needed for geodesic error
    device=DEVICE,
    k=K_EIG,
)

test_dataset = PairsDataset(test_shapes, pair_mode="all")
print(f"Test shapes: {len(test_shapes)},  Test pairs: {len(test_dataset)}")

c:\Users\giuli\OneDrive\Research\geomfum_proj\venv\Lib\site-packages\gsops\pytorch\sparse.py:21: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return _torch.sparse_csc_tensor(ccol_indices, row_indices, values, size=array.shape)


Test shapes: 2,  Test pairs: 2


## 2. Build `RobustFMNet` from the URRSM JSON config

`benchfum/_build.py` provides a fully generic JSON→Python factory.  
The config `configs/models/robust_fmnet.json` encodes **all** URRSM-exact hyperparameters:

| Hyperparameter | Value |
|---|---|
| DiffusionNet k_eig | 128 |
| DiffusionNet out_channels | 256 |
| WKS descriptor | `UrrsmWksDomain(n_domain=128)` + `L2InnerNormalizer` |
| λ (fmap regulariser) | 100 |
| γ (resolvent) | 0.5 |
| Softmax τ | 0.07 |

In [7]:
import sys

sys.path.append("../../../geomfum/")
from benchfum._build import build_model_from_json

MODEL_CONFIG = "../../../geomfum/benchfum/configs/models/robust_fmnet.json"

model = build_model_from_json(MODEL_CONFIG, device=DEVICE)
print(model)

RobustFMNet(
  (feature_extractor): DiffusionnetFeatureExtractor(
    (model): DiffusionNet(
      (first_linear): Linear(in_features=128, out_features=128, bias=True)
      (last_linear): Linear(in_features=128, out_features=256, bias=True)
      (blocks): ModuleList(
        (0-3): 4 x DiffusionNetBlock(
          (diffusion): LearnedTimeDiffusion()
          (gradient_features): SpatialGradientFeatures(
            (A_re): Linear(in_features=128, out_features=128, bias=False)
            (A_im): Linear(in_features=128, out_features=128, bias=False)
          )
          (mlp): MiniMLP(
            (miniMLP_linear_000): Linear(in_features=384, out_features=128, bias=True)
            (miniMLP_activation_000): ReLU()
            (miniMLP_dropout_001): Dropout(p=0.5, inplace=False)
            (miniMLP_linear_001): Linear(in_features=128, out_features=128, bias=True)
            (miniMLP_activation_001): ReLU()
            (miniMLP_dropout_002): Dropout(p=0.5, inplace=False)
          

## 3. Load a pretrained checkpoint

The checkpoint `ulrssm_faust.pth` was produced by the original URRSM code.  
It is a **raw DiffusionNet `state_dict`** (keys start with `first_linear`, `blocks.*`, …).  
Our `FeatureExtractor` wraps DiffusionNet as `.model`, so we add a `"model."` prefix before loading.

In [8]:
CHECKPOINT_PATH = (
    "../../../geomfum/benchfum/configs/checkpoints/Robust_FMNet/ulrssm_faust.pth"
)

raw_sd = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

# The checkpoint stores DiffusionNet weights directly;
# FeatureExtractor nests the network under .model, so we remap the keys.
remapped_sd = {"model." + k: v for k, v in raw_sd.items()}
model.feature_extractor.load_state_dict(remapped_sd)
print("Checkpoint loaded successfully.")
print(
    f"DiffusionNet: in_channels={raw_sd['first_linear.weight'].shape[1]}, "
    f"hidden={raw_sd['first_linear.weight'].shape[0]}, "
    f"out_channels={raw_sd['last_linear.weight'].shape[0]}"
)

Checkpoint loaded successfully.
DiffusionNet: in_channels=128, hidden=128, out_channels=256


## 4. Inference with `TrainedModelWrapper`

`TrainedModelWrapper` puts the model in eval mode, disables gradients,  
and provides the same `__call__` interface as classical matchers —  
so it can be dropped into any `Experiment` or `ExperimentSuite`.

In [9]:
from geomfum.learning.wrappers import TrainedModelWrapper

matcher = TrainedModelWrapper(model, device=DEVICE)

# Grab one test pair
pair = test_dataset[0]
shape_a = pair["source"]["shape"]
shape_b = pair["target"]["shape"]

result = matcher(shape_a, shape_b)
print(
    "p2p21 shape:", result.p2p21.shape
)  # [n_b] — for each vertex in B, its match in A
print("fmap12 shape:", result.fmap12.shape)  # [K_b, K_a]

p2p21 shape: torch.Size([6890])
fmap12 shape: torch.Size([128, 128])


## 5. Geodesic error evaluation

For FAUST, the shapes share the same triangulation, so ground-truth correspondences  
are the identity map: `corr_a = corr_b = [0, 1, …, N-1]`.  
We use the precomputed geodesic distance matrices from the `dist/` folder.

In [ ]:
import torch

n_verts_b = shape_b.vertices.shape[0]
corr_a = (
    pair["source"]["corr"]
    if "corr" in pair["source"]
    else torch.arange(n_verts_b, device=DEVICE)
)  # identity correspondences
corr_b = (
    pair["target"]["corr"]
    if "corr" in pair["target"]
    else torch.arange(n_verts_b, device=DEVICE)
)  # identity correspondencess

# Predicted p2p12: for each vertex i in A, what is its match in B?
# We use result.p2p21 as p2p12 here (both shapes have same topology in FAUST)
p2p12 = result.p2p21.to(DEVICE)

dist_b = pair["target"]["dist_matrix"].to(DEVICE)  # [n_b, n_b]

geo_err = torch.mean(dist_b[p2p12[corr_a], corr_b])
print(f"Geodesic error (w/o refinement): {geo_err.item():.4f}")

Geodesic error (w/o refinement): 0.0201


## 6. Test-time refinement

URRSM runs **5 gradient steps per test pair** using the unsupervised loss  
(bijectivity + orthonormality + alignment + Dirichlet energy).  
Weights are restored after each pair, so this is purely *transductive* refinement.

`build_test_time_refiner_from_json` reads `configs/training/urrsm_test_time.json`  
which has all four losses pre-configured.

In [11]:
from benchfum._build import build_test_time_refiner_from_json

TTR_CONFIG = "../../../geomfum/benchfum/configs/training/urrsm_test_time.json"

test_time_refiner = build_test_time_refiner_from_json(TTR_CONFIG, model=model)
print(
    f"TestTimeRefiner: {test_time_refiner.n_steps} steps, "
    f"lr={test_time_refiner.optimizer_config['lr']}, "
    f"grad_clip={test_time_refiner.grad_clip_norm}"
)
print("Losses:", [type(l).__name__ for l in test_time_refiner.loss_manager.losses])

TestTimeRefiner: 5 steps, lr=0.001, grad_clip=1.0
Losses: ['OrthonormalityLoss', 'BijectivityLoss', 'FmapDescriptorsSupervisionLoss', 'DirichletLoss']


In [12]:
result_refined = test_time_refiner(shape_a, shape_b)
p2p12_refined = result_refined.p2p21.to(DEVICE)

geo_err_refined = torch.mean(dist_b[p2p12_refined[corr_a], corr_b])
print(f"Geodesic error (w/o refinement): {geo_err.item():.4f}")
print(f"Geodesic error (w/  refinement): {geo_err_refined.item():.4f}")

Geodesic error (w/o refinement): 0.0201
Geodesic error (w/  refinement): 0.0200


## 7. Full evaluation loop

Iterate over all test pairs and report mean geodesic error.

In [13]:
errors_base = []
errors_refine = []

for pair in test_dataset:
    shape_a = pair["source"]["shape"]
    shape_b = pair["target"]["shape"]
    dist_b = pair["target"]["dist_matrix"].to(DEVICE)
    n_b = shape_b.vertices.shape[0]
    corr_a = torch.arange(n_b, device=DEVICE)
    corr_b = torch.arange(n_b, device=DEVICE)

    # Base inference
    res = matcher(shape_a, shape_b)
    p2p = res.p2p21.to(DEVICE)
    errors_base.append(torch.mean(dist_b[p2p[corr_a], corr_b]).item())

    # Test-time refinement
    res_r = test_time_refiner(shape_a, shape_b)
    p2p_r = res_r.p2p21.to(DEVICE)
    errors_refine.append(torch.mean(dist_b[p2p_r[corr_a], corr_b]).item())

import statistics

print(f"Mean geodesic error  (base):      {statistics.mean(errors_base):.4f}")
print(f"Mean geodesic error  (+ TTR):     {statistics.mean(errors_refine):.4f}")

Mean geodesic error  (base):      0.0200
Mean geodesic error  (+ TTR):     0.0197


## 8. Training from scratch

The URRSM training config `configs/training/urrsm.json` encodes:

| Setting | Value |
|---|---|
| Optimizer | Adam lr=1e-3 |
| Scheduler | CosineAnnealingLR (T_max=15, η_min=1e-4) |
| Epochs | 15 |
| Losses | Orthonormality + Bijectivity + FmapDescriptorSupervision (all w=1) |
| Gradient clip | 1.0 |
| Validation metric | GeodesicError (min) |

In [14]:
from geomfum.dataset.torch import PairsDataset, ShapeDataset

train_shapes = ShapeDataset(
    TRAIN_SET_PATH,
    spectral=True,
    distances=False,
    correspondences=False,
    device=DEVICE,
    k=K_EIG,
)
train_dataset = PairsDataset(train_shapes, pair_mode="all")
print(f"Train shapes: {len(train_shapes)},  Train pairs: {len(train_dataset)}")

Train shapes: 5,  Train pairs: 20


In [15]:
from benchfum._build import build_model_from_json, build_trainer_from_json

TRAINING_CONFIG = "../../../geomfum/benchfum/configs/training/urrsm.json"
CHECKPOINT_SAVE = "./urrsm_faust_best.pth"  # where to save the best checkpoint

# Build a fresh model (we don't load the pretrained weights here)
model_train = build_model_from_json(MODEL_CONFIG, device=DEVICE)

trainer = build_trainer_from_json(
    TRAINING_CONFIG,
    model=model_train,
    train_set=train_dataset,
    val_set=test_dataset,
)
# Override checkpoint path so the best model is saved
trainer.checkpoint_path = CHECKPOINT_SAVE
trainer.device = DEVICE

print("Trainer config:")
print(f"  epochs       = {trainer.epochs}")
print(f"  optimizer    = {type(trainer.optimizer).__name__}")
print(f"  scheduler    = {type(trainer.scheduler).__name__}")
print(f"  grad_clip    = {trainer.grad_clip_norm}")
print(f"  monitor      = {trainer.monitor_metric} ({trainer.mode})")
print(
    f"  train losses = {[type(l).__name__ for l in trainer.train_loss_manager.losses]}"
)
print(f"  val   losses = {[type(l).__name__ for l in trainer.val_loss_manager.losses]}")

Trainer config:
  epochs       = 15
  optimizer    = Adam
  scheduler    = CosineAnnealingLR
  grad_clip    = 1.0
  monitor      = GeodesicError (min)
  train losses = ['OrthonormalityLoss', 'BijectivityLoss', 'FmapDescriptorsSupervisionLoss']
  val   losses = ['GeodesicError']


In [18]:
# Uncomment to start training
trainer.train()

INFO:root:Epoch [1/15] - Training
Epoch 1/15 (Train): 100%|██████████| 20/20 [03:30<00:00, 10.52s/batch, Loss=334.9606]
INFO:root:Epoch [1/15] - Average Training Loss: 359.4446
Validation:   0%|          | 0/2 [00:11<?, ?batch/s]


KeyError: 'corr_a'

## 9. Load a saved checkpoint and run post-training evaluation

After training (or after copying a checkpoint from the `configs/checkpoints/` folder),  
we can load the full trainer checkpoint (which includes optimizer + scheduler states).

In [17]:
# Load model from a trainer checkpoint (produced by DeepFunctionalMapTrainer)
import os

EVAL_CHECKPOINT = (
    CHECKPOINT_SAVE  # use the checkpoint we just trained, or point elsewhere
)

if os.path.exists(EVAL_CHECKPOINT):
    from geomfum.learning.wrappers import TrainedModelWrapper

    eval_model = build_model_from_json(MODEL_CONFIG, device=DEVICE)
    eval_matcher = TrainedModelWrapper(
        eval_model, device=DEVICE, checkpoint_path=EVAL_CHECKPOINT
    )
    print("Checkpoint loaded for evaluation.")
else:
    print(f"No checkpoint found at {EVAL_CHECKPOINT} — run trainer.train() first.")

No checkpoint found at ./urrsm_faust_best.pth — run trainer.train() first.
